# Installing a Jupyter Docker Image

## Selecting an image
For reference and description of available images, see 
**[User Guide for Jupyter Docker Stacks](https://jupyter-docker-stacks.readthedocs.io)**

Jupyter images are maintained by Red Hat Quay.io.  This is the repository:  
**[Jupyter-Docker images](https://quay.io/organization/jupyter)**


## Images for Serengeti 

The docker image for scipy notebooks is used for data download, transforms and views.
There are several scipy-notebooks use for COVID data processing.  Each has a share volume for Jupyter's *home/jovyan/work* directory at 
`/home/emma/oxford/jupyter/scipy/covid_data_n/work` where $1 <= n <= 5$.

The docker image for pytorch-notebooks is used for various models.
The pytorch-notebook has a share volume for Jupyter's *home/jovyan/work* directory at `/home/emma/oxford/jupyter/scipy/covid-data/work`.

## Make sure the Docker daemon is running
- multiple ways to check docker status including:
```
sudo systemctl status docker
             or
sudo systemctl is-active docker
```
- if not active:
```
sudo systemctl start docker
```
## How to avoid using `sudo`
You can run docker with root privileges, e.g. `sudo`, or you can create a docker group and add yourself as a member.
```
 sudo groupadd docker 
 sudo usermod -aG docker ${USER}
```
If you cannot run docker without using ``sudo``, check the docker group's permissions.

## Current state of docker group and users on host

The host is hugo, emma's workstation.  It has a docker group and emma is a member.  

# Starting Jupyter server

**Important:** 
 - **``host``** is the name for the computer running the jupyter docker image - also referred to as  `outer`.
 - **``docker``** is the name for the docker container running jupyter - also referred to as  `internal or inner`.

## docker run

Start docker using the ``docker run`` command.

In this example,  we are starting a``jupyter/scipy-notebook`` image tagged ``2025-03-14`` from Quay.io   If an image with this tag is not found on the host, then it is downloaded.

The example run command below does 4 things:

1) if not on the host already, pulls the jupyter/scipy-notebook image for 2025-03-14 from quay.io.
2) Starts an ephemeral jupyter-docker container running a Jupyter Server for a scipy-notebook.
3) exposes the  internal port 8888 to the host port 10000.
4) deletes the container's file-system on exit.

 ```
 docker run -it --rm -v "${PWD}":/home/jovyan/work  -p 10000:8888 quay.io/jupyter/scipy-notebook:2025-03-14
 ```
 
 uses the following options:
 - ``-it``: makes the process interactive (i) on the current terminal (t)
 - ``-p``: defines port mappings as `<host port><jupyter-docker port>`. 
 - ``-v`` (optional):  When you define a share directory with the -v flag, entries to ``home/jovyan/work`` on jupyter-docker are saved on the host in ``${PWD}``.
 - ``--rm`` (optional):  cleanup the container and remove its file system when the container exists.

This *docker run* command starts a Jupyter Server running on a docker container.  The server has the JupyterLab frontend and exposes the server on host port 8888. The server logs appear in the terminal and include a URL to the server.

```
# Entered start.sh with args: jupyter lab

# ...

#     To access the server, open this file in a browser:
#         file:///home/jovyan/.local/share/jupyter/runtime/jpserver-7-open.html
#     Or copy and paste one of these URLs:
#         http://eca4aa01751c:8888/lab?token=d4ac9278f5f5388e88097a3a8ebbe9401be206cfa0b83099
#         http://127.0.0.1:8888/lab?token=d4ac9278f5f5388e88097a3a8ebbe9401be206cfa0b83099
```

## Help for *docker* and *docker run*

Docker commands: `docker -help`
   
Docker run options: `docker run --help`

## Common errors at startup

### Could not find/use the image

Make sure the image is available on quay.io.  If you have given an exact date for the image, it may fail because images are no longer built every day.

### Out-of-storage error during image download

You may run out of storage because the image was not found on the host.  Then docker, if given a repository address, will attempt a download.  
Images are usually over a GB.  Given a download speed of 100 Mbps, the pull takes about 5 minutes.  

If the filesystem for `/var/lib/docker/overlay2` runs out of storage you will see:

**Out of storage error:**  The download has stopped and the incompletely downloaded bits are deleted.  

**Solution:**  Remove unused images using the prune command:

    docker system prune -af

Another solution, that avoids deleting container data, is to remount the overlay directory on a filesystem allocated with more storage.

## Stopping jupyter server

If you are running the server interactively, then, from the terminal where the server is running, pressing **`Ctrl-C`** twice shuts down the Server.

If you are browsing jupyterlab, then File -> Shutodwn shuts down the server.

### Caveat:  Data deletion at shutdown.  

The docker run option "**-rm**" is a particularly dangerous option.  If you start the server using the *-rm* flag, then, at shutdown, the server delete all notebook data living on the overlay file system.  The containeris stopped and removed.  

If you did not use the *-rm* flag, then, at shutdown, the jupyter-docker container remains intact on disk for later restart or permanent deletion.  

As a rule, using the *-rm* flag is not useful when actively creating or changing a notebook.  It may be useful if the notebook is to be used without changes.

Of course, no matter what, the data on the host's shared volume remains intact.  



# Viewing a jupyter notebook via jupyterlab's URL

To access the jupyter-docker container, we use its frontend, jupyterlab.  To access jupyterlab, we make use of the port mapping defined as `<host:docker> = <10000:8888>`.  

Browsing to ``http://<hostname>:10000`` will open port 8888  on the docker container.  Jupyter lab is running on the container's 8888 port.

To access JupyterLab, you need the token.  Append the token to the URL:

    http://<hostname>:10000/?token=<token>
    
The ``<token>`` is the secret token printed in the console during startup.


## The magic token is found near the end of  jupyter startup messages

Near the end of the startup messages, there's a block of text that contains the token.  It looks like this:

```
...
[I 2025-04-28 18:08:38.268 ServerApp] Jupyter Server 2.15.0 is running at:
[I 2025-04-28 18:08:38.268 ServerApp] http://localhost:8888/lab?token=34943ebe97b4f89c667f18df9dad077fb88a590dcb07de04
[I 2025-04-28 18:08:38.268 ServerApp]     http://127.0.0.1:8888/lab?token=34943ebe97b4f89c667f18df9dad077fb88a590dcb07de04
[I 2025-04-28 18:08:38.268 ServerApp] Use Control-C to stop this server and shut down all kernels (twice to skip confirmation).
[C 2025-04-28 18:08:38.270 ServerApp] 
    
    To access the server, open this file in a browser:
        file:///home/jovyan/.local/share/jupyter/runtime/jpserver-7-open.html
    Or copy and paste one of these URLs:
        http://localhost:8888/lab?token=34943ebe97b4f89c667f18df9dad077fb88a590dcb07de04
        http://127.0.0.1:8888/lab?token=34943ebe97b4f89c667f18df9dad077fb88a590dcb07de04
  ...
```

The token is in the URLs: 34943ebe97b4f89c667f18df9dad077fb88a590dcb07de04

## **Reminder**: It's Jupyter running in a container. 

The startup messages are provided by the jupyter server, running inside docker.  Therefore, the port number is for the jupyter-docker container and will not work for a user running on the host machine.  In that case, make sure to use the host port 10000:

```
http://localhost:10000/lab?token=34943ebe97b4f89c667f18df9dad077fb88a590dcb07de04
http://127.0.0.1:10000/lab?token=34943ebe97b4f89c667f18df9dad077fb88a590dcb07de04
http://<hostname>:10000/lab?token=34943ebe97b4f89c667f18df9dad077fb88a590dcb07de04
```


# Fundamental docker commands for containers

The jupyter server runs in a docker container.  When the container goes down, so does the server - and vice versa.

### List containers

`docker ps` provides information on *running* containers

`docker ps --all` provides information for all containers - both stopped (exited) or running.

**Example:**  

    CONTAINER ID   IMAGE                                       COMMAND                  CREATED              STATUS                     PORTS     NAMES
    eca4aa01751c   quay.io/jupyter/scipy-notebook:2025-03-14   "tini -g -- start-no…"   About a minute ago   Exited (0) 5 seconds ago             silly_panini

### Start a stopped container

docker start --attach -i CONTAINER_ID or NAMES 

**Example:**  

    docker start --attach -i eca4aa01751c

### Remove a stopped container

docker remove CONTAINER_ID or NAMES

    docker rm eca4aa01751c

---

# References

1) [Complete ``docker run`` Options Explained](https://docs.docker.com/reference/cli/docker/container/run/).

2) [Supercharging AI/ML Development with JupyterLab and Docker](https://www.docker.com/blog/supercharging-ai-ml-development-with-jupyterlab-and-docker/)